## Decision Tree from Stratch

#### Libraries

In [51]:
import pandas as pd
import numpy as np
import numpy as np
import numpy.typing as npt

# interpretation of result
from sklearn.metrics import mean_squared_error, r2_score

#### Load Case 1

In [52]:
df_train1 = pd.read_csv('../data/case1/train1.csv')
df_test1 = pd.read_csv('../data/case1/test1.csv')

##### Change relevant columns

In [53]:
# speed in krpm, n
df_train1['krpm'] = df_train1['speed_rpm'] / 1000

# pressure rise across pump
df_train1['delta_p'] = (
    df_train1['outlet_pressure_bar'] - df_train1['inlet_pressure_bar']
)

# speed in krpm, n
df_test1['krpm'] = df_test1['speed_rpm'] / 1000

# pressure rise across pump
df_test1['delta_p'] = (
    df_test1['outlet_pressure_bar'] - df_train1['inlet_pressure_bar']
)


In [54]:
exclude_cols = [
    'pump_id',
    'elapsed_s',
    'operating_state',
    'inlet_pressure_bar',
    'outlet_pressure_bar',
    'flow_m3_h'
]

X_train = df_train1.drop(columns = exclude_cols)
y_train = df_train1['flow_m3_h']

X_test = df_test1.drop(columns = exclude_cols)
y_test = df_test1['flow_m3_h']

In [55]:
def gini_impurity(labels):
    '''
    compute Gini Impurity for a set of labels
    formula: gini = 1 - sum(p_1^2), where p_1 is proportion of class 1
    ''' 
    if len(labels) == 0:
        return 0

    # count occurences of each class
    unique_classes , counts = np.unique(labels , return_counts = True)
    proportions = counts / len(labels)

    # compute Gini Impurity
    gini = 1 - np.sum(proportions ** 2)
    return gini

def weighted_impurity(left_labels , right_labels , impurity_func = gini_impurity):
    ''' 
    compute weighted impurity for a split
    formula: weighted impurity = (n_left / n_total) * impurity_left + (n_right / n_total) * impurity_right
    ''' 
    n_left = len(left_labels)
    n_right = len(right_labels)
    n_total = n_left + n_right

    if n_total == 0:
        return 0

    # compute weighted average of impurities
    weighted_imp = (n_left / n_total) * impurity_func(left_labels) + (n_right / n_total) * impurity_func(right_labels)
    return weighted_imp

In [56]:
# reference Medium: https://medium.com/@darkmortal/implementing-decision-tree-regression-from-scratch-without-using-scikit-learn-1a3ebd4b2354

def w_mse(left_labels , right_labels , y):

    if len(left_labels) == 0 or len(right_labels) == 0:
        return np.inf

    left_mse = np.mean((left_labels - np.mean(left_labels)) ** 2)
    right_mse = np.mean((right_labels - np.mean(right_labels)) ** 2)

    weighted_mse = (len(left_labels) * left_mse + len(right_labels) * right_mse) / len(y)

    return weighted_mse

In [57]:
# reference: exercise session ... and Medium
# with changes to fit regression problem instead of classification problem
# changes refere

class Node:
    '''
    A node in the decision tree
    '''
    def __init__(self , feature = None , threshold = None , left = None , right = None , value = None):
        self.feature = feature          # index of feature to split on
        self.threshold = threshold      # treshold value for the split
        self.left = left                # left child node
        self.right = right              # right child node
        self.value = value              # class labels for leaf nodes

    def is_leaf(self):
        '''
        check if node is a leaf
        '''
        return self.value is not None
    
class DecisionTreeFromScratch:
    '''
    a decision tree regressor from scratch
    ''' 
    def __init__(self , max_depth = None , min_samples_split = 2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self , X , y):
        ''' 
        build the decision tree
        ''' 
        # because root exepct NumPy array
        X = np.array(X)
        y = np.array(y)

        self.root = self._build_tree(X , y , depth = 0)
        return self
    
    def _build_tree(self , X , y , depth):
        ''' 
        recursively build the decision tree
        ''' 
        n_samples , n_features = X.shape

        # stopping conditions
        if (self.max_depth is not None and depth >= self.max_depth) or \
            n_samples < self.min_samples_split or len(set(y)) == 1:
                #create a leaf node with the most common class
                leaf_value = np.mean(y)
                return Node(value = leaf_value)

        # find best split
        best_feature , best_threshold = self._best_split(X , y)

        # if not split improves tree, create a leaf node
        if best_feature is None:
            leaf_value = np.mean(y)
            return Node(value = leaf_value)

        # split the dataset
        left_indices = X[: , best_feature] <= best_threshold
        right_indices = X[: , best_feature] > best_threshold

        # recursively build left and right subtrees
        left_child = self._build_tree(X[left_indices] , y[left_indices] , depth + 1)
        right_child = self._build_tree(X[right_indices] , y[right_indices] , depth + 1)

        return Node(
            feature = best_feature,
            threshold = best_threshold,
            left = left_child,
            right = right_child
        )
    
    def _best_split(self , X , y):
        ''' 
        find the best split for the dataset
        '''
        n_samples , n_features = X.shape

        if n_samples <= 1:
            return None , None

        best_mse = float('inf')
        best_feature = None
        best_threshold = None

        # try all features
        for feature_idx in range(n_features):
            X_col = X[: , feature_idx]
            thresholds = np.unique(X_col)

            # try all possible thresholds
            for i in range(len(thresholds) - 1):
                threshold = (thresholds[i] + thresholds[i + 1]) / 2

                # split the data
                left_indices = X_col <= threshold
                right_indices = X_col > threshold

                left_y = y[left_indices]
                right_y = y[right_indices]

                # compute weighted MSE
                weight_mse = w_mse(left_y , right_y , y)
                

                # update best split if better
                if weight_mse < best_mse:
                    best_mse = weight_mse
                    best_feature = feature_idx
                    best_threshold = threshold

        return best_feature , best_threshold

    def predict(self , X):
        ''' 
        predict class labels for samples in X
        ''' 
        X = np.asarray(X)
        return np.array([self._traverse_tree(x , self.root) for x in X])
    
    def _traverse_tree(self , x , node):
        ''' 
        traverse the trees to make a prediction for single sample
        ''' 
        # if we are at leaf node, return its value
        if node.is_leaf():
            return node.value

        # otherwise, traverse left or right based on feature value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x , node.left)
        else:
            return self._traverse_tree(x , node.right)



In [58]:
for depth in [1,2,3,4,5,6,8,10]:
    for splits in [2,5,10,20,50]:
        dt = DecisionTreeFromScratch(
            max_depth = depth,
            min_samples_split = splits
        )

        dt.fit(X_train, y_train)

        y_train_pred = dt.predict(X_train)
        y_test_pred = dt.predict(X_test)

        train_mse = mean_squared_error(y_train , y_train_pred)
        test_mse = mean_squared_error(y_test , y_test_pred)

        train_r2 = r2_score(y_train, y_train_pred)
        test_r2 = r2_score(y_test, y_test_pred)

        print(
            f"Depth = {depth} and, "
            f"Minimum samples = {splits}: "
            f"Train MSE = {train_mse:.3f}, "
            f"Test MSE = {test_mse:.3f}, "
            f"Train R2 = {train_r2:.3f}, "
            f"Test R2 = {test_r2:.3f}"
        )

Depth = 1 and, Minimum samples = 2: Train MSE = 113.210, Test MSE = 134.483, Train R2 = 0.336, Test R2 = 0.077
Depth = 1 and, Minimum samples = 5: Train MSE = 113.210, Test MSE = 134.483, Train R2 = 0.336, Test R2 = 0.077
Depth = 1 and, Minimum samples = 10: Train MSE = 113.210, Test MSE = 134.483, Train R2 = 0.336, Test R2 = 0.077
Depth = 1 and, Minimum samples = 20: Train MSE = 113.210, Test MSE = 134.483, Train R2 = 0.336, Test R2 = 0.077
Depth = 1 and, Minimum samples = 50: Train MSE = 113.210, Test MSE = 134.483, Train R2 = 0.336, Test R2 = 0.077
Depth = 2 and, Minimum samples = 2: Train MSE = 85.923, Test MSE = 128.905, Train R2 = 0.496, Test R2 = 0.116
Depth = 2 and, Minimum samples = 5: Train MSE = 85.923, Test MSE = 128.905, Train R2 = 0.496, Test R2 = 0.116
Depth = 2 and, Minimum samples = 10: Train MSE = 85.923, Test MSE = 128.905, Train R2 = 0.496, Test R2 = 0.116
Depth = 2 and, Minimum samples = 20: Train MSE = 85.923, Test MSE = 128.905, Train R2 = 0.496, Test R2 = 0.116
